# FIT5196 Assessment 1 - Group029 Members 1-2 Integrated Working Copy

**Scope:** member 1 customers/products plus member 2 orders/order_items.

This is a tested handoff component, not the final six-table group submission.
Members responsible for deliveries and product_reviews must still integrate their
sections before the final Restart and Run All.

## 0. Configuration and reproducibility

All paths are relative to the project root. The same notebook can run from the project root or
from `notebooks/` without editing any student-specific absolute path.


In [ ]:
from pathlib import Path

GROUP_ID = "Group029"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "raw_package").exists():
    raise FileNotFoundError("Run from the Group029 project root or its notebooks folder.")

INPUT_DIR = PROJECT_ROOT / "raw_package" / "raw_input"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "member1_member2"
TEMPLATE_DIR = PROJECT_ROOT / "templates"
MAPPING_PATH = PROJECT_ROOT / "mapping" / "Group029_source_to_target_mapping_member1_member2.csv"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JSON_PATH = INPUT_DIR / f"{GROUP_ID}_commerce.json"
XML_PATH = INPUT_DIR / f"{GROUP_ID}_operations.xml"
DATA_DICTIONARY_PATH = PROJECT_ROOT / "raw_package" / "public_data_dictionary.csv"
PUBLIC_TEXT_CASES_PATH = TEMPLATE_DIR / "A1_public_text_test_cases.csv"

for required_path in [JSON_PATH, XML_PATH, DATA_DICTIONARY_PATH, PUBLIC_TEXT_CASES_PATH, MAPPING_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print({
    "group_id": GROUP_ID,
    "project_root": ".",
    "input_dir": str(INPUT_DIR.relative_to(PROJECT_ROOT)),
    "output_dir": str(OUTPUT_DIR.relative_to(PROJECT_ROOT)),
})

### 0.1 Environment and dependencies

Only Python's standard library and `pandas` are required by the member-1 transformation.
The submitted workflow performs no network access.


In [ ]:
import json
import platform
import sys
import unicodedata
import xml.etree.ElementTree as ET
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from Group029_text_functions import (
    build_latin_analysis,
    clean_narrative_text,
    contains_non_latin_script,
    extract_order_reference,
    extract_product_sku,
    extract_promo_code,
)
from group029_member1 import (
    CUSTOMER_COLUMNS,
    PRODUCT_COLUMNS,
    build_customers,
    build_products,
    collect_reference_ids,
    load_json_records,
    load_xml_root,
    sha256_file,
    validate_member1_tables,
    verify_manifest,
    write_outputs as write_member1_outputs,
)
from group029_member2 import (
    ORDER_COLUMNS,
    ORDER_ITEM_COLUMNS,
    REQUIRED_IDENTIFIER_COLUMNS,
    _semantic_missing_mask,
    build_member2_tables,
    validate_member2_tables,
    write_outputs as write_member2_outputs,
)

environment = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "kernel_executable_name": Path(sys.executable).name,
}
environment

## 1. Parse and profile the two sources

Both files are parsed structurally. The profile records source grains, nested/repeated objects,
candidate identifiers and the master-source decisions relevant to member 1.


### 1.1 JSON structure and profile


In [ ]:
manifest_audit = verify_manifest(PROJECT_ROOT / "raw_package")
json_data = load_json_records(JSON_PATH)

json_profile = pd.DataFrame([
    {
        "structural_path": "$.customerProfiles[]",
        "observed_records": len(json_data.get("customerProfiles", [])),
        "grain": "one source record per customer",
        "candidate_key": "customerID",
        "member1_role": "customers master",
    },
    {
        "structural_path": "$.orders[]",
        "observed_records": len(json_data.get("orders", [])),
        "grain": "one source record per order before reconciliation",
        "candidate_key": "orderID",
        "member1_role": "customer/product reference coverage",
    },
    {
        "structural_path": "$.orders[].shoppingCart[]",
        "observed_records": sum(len(order.get("shoppingCart", [])) for order in json_data.get("orders", [])),
        "grain": "one source record per order item before reconciliation",
        "candidate_key": "orderID + itemID",
        "member1_role": "product reference coverage",
    },
    {
        "structural_path": "$.productReviews[]",
        "observed_records": len(json_data.get("productReviews", [])),
        "grain": "one source record per review before reconciliation",
        "candidate_key": "reviewID",
        "member1_role": "shared source profile only",
    },
])

assert isinstance(json_data, dict)
assert manifest_audit["hash_matches"].all()
display(manifest_audit[["relative_path", "hash_matches"]])
display(json_profile)


### 1.2 XML structure and profile


In [ ]:
xml_root = load_xml_root(XML_PATH)

xml_profile = pd.DataFrame([
    {
        "structural_path": "/OperationsExport/ProductCatalogue/Product",
        "observed_records": len(xml_root.findall("./ProductCatalogue/Product")),
        "grain": "one source record per product",
        "candidate_key": "Product_ID",
        "member1_role": "products master",
    },
    {
        "structural_path": "/OperationsExport/Orders/Order",
        "observed_records": len(xml_root.findall("./Orders/Order")),
        "grain": "one source record per order before reconciliation",
        "candidate_key": "Order_ID",
        "member1_role": "customer/product reference coverage",
    },
    {
        "structural_path": "/OperationsExport/Orders/Order/Shopping_Cart/Item",
        "observed_records": len(xml_root.findall("./Orders/Order/Shopping_Cart/Item")),
        "grain": "one source record per order item before reconciliation",
        "candidate_key": "Order_ID + Item_ID",
        "member1_role": "product reference coverage",
    },
    {
        "structural_path": "/OperationsExport/ProductReviews/Review",
        "observed_records": len(xml_root.findall("./ProductReviews/Review")),
        "grain": "one source record per review before reconciliation",
        "candidate_key": "Review_ID",
        "member1_role": "shared source profile only",
    },
])

assert xml_root.tag == "OperationsExport"
display(pd.DataFrame([{"root_tag": xml_root.tag, **xml_root.attrib}]))
display(xml_profile)


### 1.3 Source comparison and assumptions

Orders occur in both sources with different field names and representations.
They are standardised independently, compared field by field by business key,
and only then collapsed to canonical rows. Reported arithmetic is treated as
validation evidence; submitted arithmetic is independently recomputed.

In [ ]:
source_decisions = pd.DataFrame([
    {
        "target_or_issue": "customers",
        "JSON evidence": "$.customerProfiles[] contains the 20 published customer attributes",
        "XML evidence": "orders contain customer references but no customer master",
        "decision": "JSON customerProfiles is the master; order IDs are referential checks only",
    },
    {
        "target_or_issue": "products",
        "JSON evidence": "shoppingCart contains product references but no product master",
        "XML evidence": "ProductCatalogue/Product contains the 21 published product attributes",
        "decision": "XML ProductCatalogue is the master; cart IDs are referential checks only",
    },
    {
        "target_or_issue": "orders and order_items",
        "JSON evidence": "$.orders[].header plus repeated $.orders[].shoppingCart[]",
        "XML evidence": "/OperationsExport/Orders/Order/Header plus repeated Shopping_Cart/Item",
        "decision": "standardise each source, compare by order_id/order_item_id, stop on conflict, then retain one canonical row",
    },
    {
        "target_or_issue": "dates, flags and money",
        "JSON evidence": "ISO timestamps, native booleans and typed numerics",
        "XML evidence": "DD/MM/YYYY timestamps, Y/N flags, AUD labels, commas and percent signs",
        "decision": "parse exact source formats; emit published target formats and typed values",
    },
    {
        "target_or_issue": "order arithmetic",
        "JSON evidence": "reported line/order/GST/total values",
        "XML evidence": "reported line/order/GST/total values with AUD formatting",
        "decision": "recompute from canonical quantity/unit_price in the published sequence; use reported fields only for 0.01-tolerance validation",
    },
])
display(source_decisions)

## 2. Source-to-target mapping

The official 111-row template is retained. Member 1 completes only the 20 `customers` rows and
21 `products` rows; the other 70 rows remain blank for their owners. Structural source paths are
blank when that source does not supply the target field.


In [ ]:
mapping = pd.read_csv(MAPPING_PATH, keep_default_na=False, dtype=str)
integrated_mapping = mapping[
    mapping["output_table"].isin(["customers", "products", "orders", "order_items"])
].copy()
required_mapping_columns = [
    "source_format",
    "transformation_or_derivation",
    "overlap_or_conflict_rule",
    "notebook_evidence",
]

assert len(mapping) == 111
assert len(integrated_mapping) == 70
assert integrated_mapping["mapping_id"].str.startswith("MAP-").all()
assert integrated_mapping[required_mapping_columns].ne("").all().all()

mapping_status = integrated_mapping.groupby("output_table", as_index=False).agg(
    completed_rows=("mapping_id", "size"),
    first_mapping_id=("mapping_id", "first"),
    last_mapping_id=("mapping_id", "last"),
)
display(mapping_status)
display(integrated_mapping[integrated_mapping["output_table"].isin(["orders", "order_items"])].head(8))

## 3. Text and regex functions

`products.product_description_clean` depends on the shared published cleaning contract. The
module also exposes the other five fixed functions so the group has one reviewable interface.
The review-table owner must still integrate these functions into `product_reviews`.


### 3.1 Cleaning and extraction implementation


In [ ]:
text_function_contract = pd.DataFrame([
    {"function": "clean_narrative_text", "member1_use": "product_description_clean"},
    {"function": "extract_order_reference", "member1_use": "shared interface only"},
    {"function": "extract_product_sku", "member1_use": "shared interface only"},
    {"function": "extract_promo_code", "member1_use": "shared interface only"},
    {"function": "build_latin_analysis", "member1_use": "shared interface only"},
    {"function": "contains_non_latin_script", "member1_use": "shared interface only"},
])
display(text_function_contract)
print("Implementation:", str((SRC_DIR / "Group029_text_functions.py").relative_to(PROJECT_ROOT)))


### 3.2 Public and student-designed tests


In [ ]:
text_functions = {
    "clean_narrative_text": clean_narrative_text,
    "extract_order_reference": extract_order_reference,
    "extract_product_sku": extract_product_sku,
    "extract_promo_code": extract_promo_code,
    "build_latin_analysis": build_latin_analysis,
    "contains_non_latin_script": contains_non_latin_script,
}

public_cases = pd.read_csv(PUBLIC_TEXT_CASES_PATH, keep_default_na=False, dtype=str)
public_results = []
for case in public_cases.itertuples(index=False):
    observed = text_functions[case.function](case.input_value)
    observed_text = str(observed) if isinstance(observed, bool) else observed
    public_results.append({
        "case_id": case.case_id,
        "function": case.function,
        "expected_output": case.expected_output,
        "observed_output": observed_text,
        "status": "PASS" if observed_text == case.expected_output else "FAIL",
    })
public_results = pd.DataFrame(public_results)

student_cases = pd.DataFrame([
    {"case_id": "M1-EDGE-01", "observed": extract_order_reference("XHORD123456"), "expected": "NaN"},
    {"case_id": "M1-EDGE-02", "observed": extract_product_sku("SKU-ABC123-extra"), "expected": "NaN"},
    {"case_id": "M1-EDGE-03", "observed": extract_promo_code("B3SAVE-240"), "expected": "NaN"},
    {"case_id": "M1-UNICODE-01", "observed": clean_narrative_text("Café 包装很好"), "expected": "café 包装很好"},
    {"case_id": "M1-MISSING-01", "observed": clean_narrative_text(None), "expected": "NaN"},
])
student_cases["status"] = student_cases["observed"].eq(student_cases["expected"]).map({True: "PASS", False: "FAIL"})

assert public_results["status"].eq("PASS").all()
assert student_cases["status"].eq("PASS").all()
display(public_results)
display(student_cases)


## 4. Build the six standardised relational tables

This working copy executes only member 1's two assigned target tables. The remaining table cells
are preserved as explicit integration boundaries for the other members.


### 4.1 `orders`

In [ ]:
member2_tables = build_member2_tables(json_data, xml_root)
orders = member2_tables["orders"]

assert orders.columns.tolist() == ORDER_COLUMNS
assert orders["order_id"].is_unique
for required_id in REQUIRED_IDENTIFIER_COLUMNS["orders"]:
    assert not _semantic_missing_mask(orders[required_id]).any(), required_id
display(member2_tables["source_profile"])
display(orders.head())

### 4.2 `order_items`

In [ ]:
order_items = member2_tables["order_items"]

assert order_items.columns.tolist() == ORDER_ITEM_COLUMNS
assert order_items["order_item_id"].is_unique
for required_id in REQUIRED_IDENTIFIER_COLUMNS["order_items"]:
    assert not _semantic_missing_mask(order_items[required_id]).any(), required_id
display(order_items.head())

### 4.3 `customers`

**Grain:** one row per customer.  
**Master source:** JSON `$.customerProfiles[]`.  
Identifiers and postcodes remain strings so leading zeroes cannot be lost.


In [ ]:
customers = build_customers(json_data)
customer_field_profile = pd.DataFrame({
    "dtype": customers.dtypes.astype(str),
    "missing": customers.isna().sum(),
    "distinct": customers.nunique(dropna=False),
})

print("customers shape:", customers.shape)
display(customers.head(3))
display(customer_field_profile)


### 4.4 `deliveries` — pending integration from the assigned owner


In [ ]:
print("Member 1 does not generate deliveries in this working copy.")


### 4.5 `products`

**Grain:** one row per product.  
**Master source:** XML `/OperationsExport/ProductCatalogue/Product`.  
Dates, AUD values and Y/N flags are parsed exactly; product narrative uses the shared text contract.


In [ ]:
products = build_products(xml_root)
product_field_profile = pd.DataFrame({
    "dtype": products.dtypes.astype(str),
    "missing": products.isna().sum(),
    "distinct": products.nunique(dropna=False),
})

print("products shape:", products.shape)
display(products.head(3))
display(product_field_profile)


### 4.6 `product_reviews` — pending integration from the assigned owner


In [ ]:
print("Member 1 does not generate product_reviews in this working copy.")


## 5. Reconcile overlap and verify relationships

The transaction sources are compared only after target normalisation. The
conflict tables identify business key, target field, sources and observed
values. Canonical selection occurs only when these tables are empty.

In [ ]:
reference_ids = collect_reference_ids(json_data, xml_root)
source_counts = {
    "customers_input": len(json_data["customerProfiles"]),
    "products_input": len(xml_root.findall("./ProductCatalogue/Product")),
}

reconciliation_evidence = member2_tables["source_profile"].copy()
reconciliation_evidence["field_conflicts"] = [
    len(member2_tables["order_conflicts"]),
    len(member2_tables["item_conflicts"]),
]
assert reconciliation_evidence["field_conflicts"].eq(0).all()
display(reconciliation_evidence)

## 6. Validation register

Each executed check has a stable `VAL-...` ID, observed result, PASS/FAIL status, evidence and an
interpretation or resolution. Counts are derived from the allocated sources rather than hard-coded
as canonical answers.


### 6.1 Schema, type and missing-value checks (`VAL-SCHEMA-...`, `VAL-TYPE-...`, `VAL-MISS-...`)


In [ ]:
member1_validation = validate_member1_tables(
    customers,
    products,
    reference_ids,
    source_counts=source_counts,
)
member2_validation = validate_member2_tables(
    member2_tables,
    customers,
    products,
)
validation = pd.concat(
    [member1_validation, member2_validation], ignore_index=True
)
schema_checks = validation[validation["area"].isin(["schema", "missingness"])]
assert validation["status"].eq("PASS").all()
display(schema_checks.drop(columns="passed"))

The executed table above is the observed result/status/interpretation record for member-1 schema and missingness checks.


### 6.2 Primary- and foreign-key checks (`VAL-PK-...`, `VAL-FK-...`)


In [ ]:
key_checks = validation[validation["area"].isin(["keys", "relationships"])]
display(key_checks.drop(columns="passed"))

The relationship checks include real anti-joins from canonical orders/items to
member-1 customers/products. They also verify every item resolves to an order.

### 6.3 Source coverage and row-flow checks (`VAL-FLOW-...`)


In [ ]:
flow_checks = validation[validation["area"].eq("flow")]
display(flow_checks.drop(columns="passed"))

The observed source counts, unique-key counts and overlap counts are calculated
from the allocated files. They are evidence, not hard-coded pipeline inputs.

### 6.4 Arithmetic checks (`VAL-ARITH-...`)

In [ ]:
arithmetic_checks = validation[validation["area"].eq("arithmetic")]
display(arithmetic_checks.drop(columns="passed"))

The checks independently recompute line revenue, order price, included GST and
the discounted-plus-delivery total. Source-reported monetary values are compared
with the required absolute tolerance of 0.01.

### 6.5 Temporal checks (`VAL-TIME-...`, `VAL-DATE-...`)


In [ ]:
temporal_checks = validation[
    validation["validation_id"].str.startswith(("VAL-TIME-", "VAL-DATE-"))
]
display(temporal_checks.drop(columns="passed"))

Member 1 verifies ISO output dates and agreement between each product launch date and launch year.


### 6.6 Text and Unicode checks (`VAL-TEXT-...`)


In [ ]:
text_checks = validation[validation["area"].eq("text")]
display(text_checks.drop(columns="passed"))

The order customer note and promotion code reuse the same published text module
tested in Section 3; no competing member-specific text implementation is used.

### 6.7 Literal `NaN` reminder

For prescribed missing narrative outputs, `NaN` means the three literal characters. CSV round-trip
validation therefore uses `keep_default_na=False`.


In [ ]:
literal_nan_check = pd.DataFrame([
    {
        "field": "orders.coupon_code",
        "literal_NaN_count": int(orders["coupon_code"].eq("NaN").sum()),
        "pandas_missing_count": int(orders["coupon_code"].isna().sum()),
        "empty_string_count": int(orders["coupon_code"].eq("").sum()),
    },
    {
        "field": "orders.promo_code",
        "literal_NaN_count": int(orders["promo_code"].eq("NaN").sum()),
        "pandas_missing_count": int(orders["promo_code"].isna().sum()),
        "empty_string_count": int(orders["promo_code"].eq("").sum()),
    },
])
assert literal_nan_check["pandas_missing_count"].eq(0).all()
assert literal_nan_check["empty_string_count"].eq(0).all()
display(literal_nan_check)

## 7. Export member-1/member-2 CSV candidates

This integrated working copy exports customers, products, orders and order_items
to one shared candidate directory. Deliveries and product_reviews remain pending.

In [ ]:
customers_path, products_path = write_member1_outputs(customers, products, OUTPUT_DIR)
orders_path, items_path, member2_validation_path = write_member2_outputs(
    orders, order_items, member2_validation, OUTPUT_DIR
)

customers_round_trip = pd.read_csv(
    customers_path, keep_default_na=False,
    dtype={"customer_id": "string", "home_postcode": "string"},
)
products_round_trip = pd.read_csv(
    products_path, keep_default_na=False,
    dtype={"product_id": "string", "product_sku": "string"},
)
orders_round_trip = pd.read_csv(
    orders_path, keep_default_na=False,
    dtype={"order_id": "string", "customer_id": "string", "coupon_code": "string", "promo_code": "string"},
)
items_round_trip = pd.read_csv(
    items_path, keep_default_na=False,
    dtype={"order_item_id": "string", "order_id": "string", "product_id": "string"},
)

assert customers_round_trip.columns.tolist() == CUSTOMER_COLUMNS
assert products_round_trip.columns.tolist() == PRODUCT_COLUMNS
assert orders_round_trip.columns.tolist() == ORDER_COLUMNS
assert items_round_trip.columns.tolist() == ORDER_ITEM_COLUMNS
assert orders_round_trip["coupon_code"].tolist() == orders["coupon_code"].tolist()

export_summary = pd.DataFrame([
    {"table": "customers", "rows": len(customers_round_trip), "columns": len(CUSTOMER_COLUMNS), "relative_path": str(customers_path.relative_to(PROJECT_ROOT))},
    {"table": "products", "rows": len(products_round_trip), "columns": len(PRODUCT_COLUMNS), "relative_path": str(products_path.relative_to(PROJECT_ROOT))},
    {"table": "orders", "rows": len(orders_round_trip), "columns": len(ORDER_COLUMNS), "relative_path": str(orders_path.relative_to(PROJECT_ROOT))},
    {"table": "order_items", "rows": len(items_round_trip), "columns": len(ORDER_ITEM_COLUMNS), "relative_path": str(items_path.relative_to(PROJECT_ROOT))},
])
display(export_summary)

## 8. Final reproducibility record

The record below is generated during the same top-to-bottom run. A successful execution recreates
the two candidate CSVs, mapping evidence and validation results without manual data edits.


In [ ]:
reproducibility_record = pd.DataFrame([{
    "group_id": GROUP_ID,
    "run_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "manifest_files_verified": int(manifest_audit["hash_matches"].sum()),
    "member1_member2_mapping_rows_complete": len(integrated_mapping),
    "validation_checks_passed": int(validation["status"].eq("PASS").sum()),
    "validation_checks_failed": int(validation["status"].eq("FAIL").sum()),
    "public_text_cases_passed": int(public_results["status"].eq("PASS").sum()),
    "orders_csv_sha256": sha256_file(orders_path),
    "order_items_csv_sha256": sha256_file(items_path),
    "members_1_2_scope_complete": True,
    "final_six_table_group_integration_complete": False,
}])
assert reproducibility_record.loc[0, "validation_checks_failed"] == 0
display(reproducibility_record.T)